# 02 · Bronze: load raw EPA files into Delta tables
Reads the original CSV files from the landing Volume, keeps only Texas water systems
and adds lineage columns. Values are not changed.


In [0]:
landing = "/Volumes/workspace/sdwa_bronze/landing/sdwa"
release = max(f.name.strip("/") for f in dbutils.fs.ls(landing))
release_path = f"{landing}/{release}"
print(release_path)


In [0]:
from pyspark.sql import functions as F

def load_bronze(file_name, table_name, texas_only=True):
    df = (
        spark.read.option("header", True)
        .option("escape", '"')
        .csv(f"{release_path}/{file_name}")
    )
    if texas_only:
        df = df.filter(F.col("PWSID").startswith("TX"))
    df = (
        df.withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_release", F.lit(release))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    full_name = f"workspace.sdwa_bronze.{table_name}"
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(full_name)
    print(f"{table_name}: {spark.table(full_name).count():,} rows")

load_bronze("SDWA_PUB_WATER_SYSTEMS.csv", "pub_water_systems")
load_bronze("SDWA_VIOLATIONS_ENFORCEMENT.csv", "violations_enforcement")
load_bronze("SDWA_GEOGRAPHIC_AREAS.csv", "geographic_areas")
load_bronze("SDWA_REF_CODE_VALUES.csv", "ref_code_values", texas_only=False)

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.sdwa_bronze.pub_water_systems)                AS system_rows,
  (SELECT COUNT(DISTINCT PWSID) FROM workspace.sdwa_bronze.pub_water_systems)   AS distinct_systems,
  (SELECT COUNT(*) FROM workspace.sdwa_bronze.violations_enforcement)           AS violation_rows,
  (SELECT COUNT(DISTINCT PWSID, VIOLATION_ID)
     FROM workspace.sdwa_bronze.violations_enforcement)                         AS distinct_violations

In [0]:
%sql
DESCRIBE workspace.sdwa_bronze.violations_enforcement

In [0]:
%sql
SELECT PWSID, VIOLATION_ID, NON_COMPL_PER_BEGIN_DATE, NON_COMPL_PER_END_DATE,
       CALCULATED_RTC_DATE, VIOLATION_CATEGORY_CODE, IS_HEALTH_BASED_IND,
       VIOLATION_STATUS, CONTAMINANT_CODE, ENFORCEMENT_ID, ENFORCEMENT_DATE, ENF_ACTION_CATEGORY
FROM workspace.sdwa_bronze.violations_enforcement
LIMIT 10

In [0]:
%sql
SELECT VIOLATION_CATEGORY_CODE, IS_HEALTH_BASED_IND, VIOLATION_STATUS,
       COUNT(DISTINCT PWSID, VIOLATION_ID) AS violations
FROM workspace.sdwa_bronze.violations_enforcement
GROUP BY ALL
ORDER BY violations DESC